In [1]:
import sys
import os
import time
import datetime

import mlflow
import numpy as np
import pandas as pd
import catboost as cb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as auc
sys.path.append(os.path.abspath("../src"))

from models_en import run_training_pipeline
from postgres_connection import connection
from Preprocessing import Preprocessing
from utils import create_submission

In [2]:
%reload_ext autoreload
%autoreload 2

In [3]:
df_description = connection("dwh.homecredit_columns_description;")
bureau = connection("dwh.bureau")
previous_application = connection("dwh.previous_application")
POS_CASH_balance = connection("dwh.POS_CASH_balance")
credit_card_balance = connection("dwh.credit_card_balance")
installments_payments = connection("dwh.installments_payments")
bureau = connection("dwh.bureau")
df_test = connection("dwh.application_test;")
df_train = connection("dwh.application_train;")

# Hypotheses


![Data model](before.jpg)

## Start

In [4]:
df_copy_final = df_train.copy()
#run_training_pipeline(df_copy_final)

## Hypothesis 1

Гипотеза: Клиенты с высокой кредитной нагрузкой относительно дохода более склонны к дефолту.

In [5]:
df_copy_final['credit_income_ratio'] = df_copy_final['amt_credit'] / df_copy_final['amt_income_total']

# 2. Создаем столбец annuity_income_ratio
df_copy_final['annuity_income_ratio'] = df_copy_final['amt_annuity'] / df_copy_final['amt_income_total']

# 3. Создаем столбец credit_term
df_copy_final['credit_term'] = df_copy_final['amt_credit'] / df_copy_final['amt_annuity']

In [6]:
df_copy_final[['credit_income_ratio', 'annuity_income_ratio', 'credit_term']].describe()

,credit_income_ratio,annuity_income_ratio,credit_term
count,307511.000000,307499.000000,307499.000000
mean,3.957570,0.180930,21.612322
std,2.689728,0.094574,7.823823
min,0.004808,0.000224,8.036674
25%,2.018667,0.114782,15.614496
50%,3.265067,0.162833,20.000000
75%,5.159880,0.229067,27.099985
max,84.736842,1.875965,45.305079


## Hypothesis 2

Гипотеза: Возраст и стаж работы влияют на надежность клиента, но нелинейно или во взаимодействии с другими факторами.


In [7]:
# 1. Создаем столбец 'age' (возраст в годах)
# Делим на -365, так как days_birth отрицательный
df_copy_final['age'] = df_copy_final['days_birth'] / -365

# 2. Создаем столбец 'years_employed' (стаж в годах)
# Делим на 365, так как очищенный 'days_employed' уже положительный (или NaN)
df_copy_final['years_employed'] = df_copy_final['days_employed'] / 365

# 3. Создаем столбец 'employed_birth_ratio' (доля рабочей жизни)
# Используем модуль от days_birth и очищенный days_employed
# abs() берет модуль от отрицательного days_birth
df_copy_final['employed_birth_ratio'] = df_copy_final['days_employed'] / df_copy_final['days_birth'].abs()

# 4. Создаем полиномиальные признаки (квадраты)
# Возводим в квадрат возраст и стаж
df_copy_final['age_sq'] = df_copy_final['age'] ** 2
df_copy_final['years_employed_sq'] = df_copy_final['years_employed'] ** 2 # NaN при возведении в квадрат останется NaN

# 5. Создаем признак взаимодействия
# Перемножаем возраст и стаж
df_copy_final['age_emp_interaction'] = df_copy_final['age'] * df_copy_final['years_employed'] # NaN при умножении останется NaN


In [8]:
df_copy_final[['age', 'years_employed', 'employed_birth_ratio']].describe()

,age,years_employed,employed_birth_ratio
count,307511.000000,307511.000000,307511.000000
mean,43.936973,174.835742,2.920135
std,11.956133,387.056895,6.627098
min,20.517808,-49.073973,-0.728811
25%,34.008219,-7.561644,-0.191000
50%,43.150685,-3.323288,-0.088645
75%,53.923288,-0.791781,-0.021559
max,69.120548,1000.665753,47.489663


## Hypothesis 3

Гипотеза: Владение недвижимостью и автомобилем снижает риск дефолта.

In [9]:
df_copy_final['flag_own_car'] = (df_copy_final['flag_own_car'] == 'Y').astype(int)
df_copy_final['flag_own_realty'] = (df_copy_final['flag_own_realty'] == 'Y').astype(int)

# 2. Создаем признак взаимодействия 'owns_car_and_realty'
df_copy_final['owns_car_and_realty'] = df_copy_final['flag_own_car'] * df_copy_final['flag_own_realty']

In [10]:
df_copy_final[['flag_own_car', 'flag_own_realty', 'owns_car_and_realty']].sample(10)

,flag_own_car,flag_own_realty,owns_car_and_realty
210692,0,1,0
237728,0,1,0
203919,0,1,0
115863,0,0,0
228601,1,1,1
248668,0,0,0
210397,0,1,0
287461,1,1,1
5564,1,1,1
252675,0,0,0


## Hypothesis 4

Гипотеза: Плохая кредитная история в бюро (bureau.csv) является сильным предиктором дефолта.

In [11]:
bureau_agg_general = bureau.groupby('sk_id_curr').agg(
    bureau_loan_count = ('sk_id_bureau', 'count'),
    bureau_loan_types_count = ('credit_type', 'nunique'),
    bureau_avg_prolong_count = ('cnt_credit_prolong', 'mean'),
    bureau_total_debt_sum = ('amt_credit_sum_debt', 'sum'),
    bureau_max_days_overdue = ('credit_day_overdue', 'max'),
    bureau_overdue_loan_count = ('amt_credit_sum_overdue', lambda x: (x > 0).sum())
).reset_index()

# --- ДОБАВЛЕННЫЙ ШАГ: Удаление существующих колонок перед merge ---
# Список колонок, которые мы создали в bureau_agg_general (кроме ключа 'sk_id_curr')
cols_to_add = [col for col in bureau_agg_general.columns if col != 'sk_id_curr']

# Находим, какие из этих колонок УЖЕ ЕСТЬ в df_copy_final
cols_to_drop = [col for col in cols_to_add if col in df_copy_final.columns]

# Если такие колонки нашлись, удаляем их из df_copy_final
if cols_to_drop:
    # print(f"Удаляем существующие столбцы перед merge: {cols_to_drop}") # Можно раскомментировать для информации
    df_copy_final = df_copy_final.drop(columns=cols_to_drop)
# -----------------------------------------------------------------

# 2. Присоединяем (merge) агрегированные данные к df_copy_final
#    Теперь конфликта имен быть не должно
df_copy_final = df_copy_final.merge(bureau_agg_general, on='sk_id_curr', how='left')

# 3. Обработка пропусков (NaN) после мержа
bureau_general_agg_cols = [
    'bureau_loan_count', 'bureau_loan_types_count', 'bureau_avg_prolong_count',
    'bureau_total_debt_sum', 'bureau_max_days_overdue', 'bureau_overdue_loan_count'
]
for col in bureau_general_agg_cols:
     if col in df_copy_final.columns:
        df_copy_final[col] = df_copy_final[col].fillna(0)
        if col == 'bureau_max_days_overdue':
             df_copy_final[col] = df_copy_final[col].clip(lower=0)

In [12]:
df_copy_final[['sk_id_curr'] + bureau_general_agg_cols].head()


,sk_id_curr,bureau_loan_count,bureau_loan_types_count,bureau_avg_prolong_count,bureau_total_debt_sum,bureau_max_days_overdue,bureau_overdue_loan_count
0,219241,7.0,2.0,0.0,1040012.250,0.0,0.0
1,219242,1.0,1.0,0.0,0.000,0.0,0.0
2,219244,0.0,0.0,0.0,0.000,0.0,0.0
3,219246,6.0,2.0,0.0,758832.435,0.0,0.0
4,219248,2.0,1.0,0.0,216004.360,0.0,0.0


## Hypothesis 5

Гипотеза: Опыт взаимодействия с Home Credit в прошлом (previous_application.csv) влияет на текущий риск.

In [13]:
prev_app_agg = previous_application.groupby('sk_id_curr').agg(
    # Общее количество и количество по статусам
    prev_app_count = ('sk_id_prev', 'count'),
    prev_app_approved_count = ('name_contract_status', lambda x: (x == 'Approved').sum()),
    prev_app_refused_count = ('name_contract_status', lambda x: (x == 'Refused').sum()),

    # Рассчитываем СУММЫ по одобренным заявкам, чтобы позже найти среднее.
    # Используем .loc для корректного выбора строк внутри лямбды по индексу группы x.index
    prev_app_approved_sum_credit = ('amt_credit', lambda x: previous_application.loc[x.index, 'amt_credit'][previous_application.loc[x.index, 'name_contract_status'] == 'Approved'].sum()),
    prev_app_approved_sum_annuity = ('amt_annuity', lambda x: previous_application.loc[x.index, 'amt_annuity'][previous_application.loc[x.index, 'name_contract_status'] == 'Approved'].sum())

).reset_index() # Возвращаем sk_id_curr из индекса в столбец

# 2. Удаляем существующие колонки из df_copy_final перед merge (если они есть)
#    Это предотвратит MergeError при повторном запуске ячейки
cols_to_add_prev = [col for col in prev_app_agg.columns if col != 'sk_id_curr']
cols_to_drop_prev = [col for col in cols_to_add_prev if col in df_copy_final.columns]
if cols_to_drop_prev:
    df_copy_final = df_copy_final.drop(columns=cols_to_drop_prev)

# 3. Присоединяем (merge) агрегированные данные к df_copy_final
df_copy_final = df_copy_final.merge(prev_app_agg, on='sk_id_curr', how='left')

# 4. Вычисляем производные признаки (доля и средние) ПОСЛЕ merge
#    Это безопаснее, т.к. мы можем обработать деление на ноль (если prev_app_count или prev_app_approved_count равны 0)

# Доля одобренных заявок
# np.divide обрабатывает деление на ноль, можно заменить результат (inf) на NaN, затем на 0
df_copy_final['prev_app_approved_rate'] = (
    df_copy_final['prev_app_approved_count'] / df_copy_final['prev_app_count']
)
# Заменяем inf (деление на 0) и -inf на NaN, затем все NaN на 0
df_copy_final['prev_app_approved_rate'] = df_copy_final['prev_app_approved_rate'].replace([np.inf, -np.inf], np.nan).fillna(0)


# Средняя сумма кредита по одобренным
df_copy_final['prev_app_approved_avg_credit'] = (
    df_copy_final['prev_app_approved_sum_credit'] / df_copy_final['prev_app_approved_count']
)
df_copy_final['prev_app_approved_avg_credit'] = df_copy_final['prev_app_approved_avg_credit'].replace([np.inf, -np.inf], np.nan).fillna(0)


# Средняя сумма аннуитета по одобренным
df_copy_final['prev_app_approved_avg_annuity'] = (
    df_copy_final['prev_app_approved_sum_annuity'] / df_copy_final['prev_app_approved_count']
)
df_copy_final['prev_app_approved_avg_annuity'] = df_copy_final['prev_app_approved_avg_annuity'].replace([np.inf, -np.inf], np.nan).fillna(0)


# 5. Обработка пропусков (NaN) для всех добавленных/вычисленных столбцов
#    Заполняем нулями столбцы, которые пришли из merge (если sk_id_curr не было в previous_application)
#    или были вычислены (если были NaN после деления)
prev_app_final_cols = [
    'prev_app_count', 'prev_app_approved_count', 'prev_app_refused_count',
    'prev_app_approved_sum_credit', 'prev_app_approved_sum_annuity', # Промежуточные суммы
    'prev_app_approved_rate', 'prev_app_approved_avg_credit', 'prev_app_approved_avg_annuity' # Финальные признаки
]
for col in prev_app_final_cols:
     if col in df_copy_final.columns:
        df_copy_final[col] = df_copy_final[col].fillna(0)

In [14]:
df_copy_final[['prev_app_count', 'prev_app_count', 'prev_app_refused_count', 'prev_app_approved_rate']].sample(5)


,prev_app_count,prev_app_count,prev_app_refused_count,prev_app_approved_rate
50144,8.0,8.0,0.0,0.875000
113882,5.0,5.0,0.0,0.400000
250479,2.0,2.0,0.0,1.000000
223826,1.0,1.0,0.0,1.000000
154416,3.0,3.0,0.0,0.333333


In [15]:
#run_training_pipeline(df_copy_final)

## Hypothesis 6

Гипотеза: Платежная дисциплина по предыдущим кредитам в Home Credit (installments_payments.csv) предсказывает будущую дисциплину.

In [16]:

# --- Реализация Гипотезы 6 (Минимальная версия, без импортов) ---

# 1. Добавляем sk_id_curr в installments_payments
prev_app_ids = previous_application[['sk_id_prev', 'sk_id_curr']]
inst_merged = installments_payments.merge(prev_app_ids, on='sk_id_prev', how='left')
# del prev_app_ids # Опционально

# 2. Обработка возможных суффиксов _x, _y для sk_id_curr (НЕОБХОДИМОЕ УСЛОВИЕ)
if 'sk_id_curr_y' in inst_merged.columns and 'sk_id_curr_x' in inst_merged.columns:
    inst_merged['sk_id_curr'] = inst_merged['sk_id_curr_y']
    inst_merged = inst_merged.drop(columns=['sk_id_curr_x', 'sk_id_curr_y'])
# Если колонки sk_id_curr нет (ни с суффиксами, ни без), последующий код вызовет KeyError или NameError

# 3. Удаляем строки, где sk_id_curr остался NaN после merge
#    Предполагаем, что sk_id_curr теперь существует, иначе будет KeyError
inst_merged = inst_merged.dropna(subset=['sk_id_curr'])
# inst_merged['sk_id_curr'] = inst_merged['sk_id_curr'].astype(int) # Опционально

# 4. Создаем признаки на уровне каждого платежа
inst_merged['payment_diff'] = inst_merged['amt_payment'] - inst_merged['amt_instalment']
inst_merged['days_late'] = inst_merged['days_entry_payment'] - inst_merged['days_instalment']
inst_merged['paid_late_flag'] = (inst_merged['days_late'] > 0).astype(int)
inst_merged['underpaid_flag'] = (inst_merged['payment_diff'] < -0.001).astype(int)
inst_merged['paid_on_time_flag'] = (inst_merged['days_late'] <= 0).astype(int)

# 5. Группируем по sk_id_curr и агрегируем
#    Предполагаем, что все исходные колонки существуют, иначе будет KeyError
inst_agg = inst_merged.groupby('sk_id_curr').agg(
    inst_payment_diff_mean = ('payment_diff', 'mean'),
    inst_payment_diff_max = ('payment_diff', 'max'),
    inst_payment_diff_sum = ('payment_diff', 'sum'),
    inst_days_late_mean = ('days_late', 'mean'),
    inst_days_late_max = ('days_late', 'max'),
    inst_days_late_sum = ('days_late', 'sum'),
    inst_late_payment_count = ('paid_late_flag', 'sum'),
    inst_underpaid_count = ('underpaid_flag', 'sum'),
    inst_paid_on_time_count = ('paid_on_time_flag', 'sum'),
    inst_total_payment_count = ('sk_id_prev', 'count')
).reset_index()

# del inst_merged # Опционально

# 6. Удаляем существующие колонки из df_copy_final перед финальным merge
#    Используем errors='ignore' чтобы не проверять наличие колонок явно
cols_to_add_inst = [col for col in inst_agg.columns if col != 'sk_id_curr']
df_copy_final = df_copy_final.drop(columns=cols_to_add_inst, errors='ignore')

# 7. Присоединяем агрегированные данные к df_copy_final
df_copy_final = df_copy_final.merge(inst_agg, on='sk_id_curr', how='left')

# 8. Вычисляем долю платежей вовремя ПОСЛЕ merge
#    Предполагаем, что колонки существуют и np импортирован, иначе будет KeyError или NameError
df_copy_final['inst_paid_on_time_rate'] = np.divide(
    df_copy_final['inst_paid_on_time_count'],
    df_copy_final['inst_total_payment_count']
)

# 9. Обработка пропусков (NaN) для всех новых столбцов
#    Предполагаем, что все эти колонки были успешно добавлены/созданы
#    Если какой-то колонки нет, fillna вызовет ошибку KeyError
inst_final_cols = [
    'inst_payment_diff_mean', 'inst_payment_diff_max', 'inst_payment_diff_sum',
    'inst_days_late_mean', 'inst_days_late_max', 'inst_days_late_sum',
    'inst_late_payment_count', 'inst_underpaid_count',
    'inst_paid_on_time_count', 'inst_total_payment_count',
    'inst_paid_on_time_rate'
]
for col in inst_final_cols:
     df_copy_final[col] = df_copy_final[col].fillna(0)
     if col == 'inst_days_late_max':
          df_copy_final[col] = df_copy_final[col].clip(lower=0)



In [17]:
# Список некоторых ключевых столбцов из Гипотезы 6 для проверки
hyp6_cols_to_sample = [
    'inst_late_payment_count',   # Кол-во опоздавших платежей
    'inst_underpaid_count',      # Кол-во недоплаченных платежей
    'inst_days_late_mean',       # Среднее опоздание в днях
    'inst_days_late_max',        # Макс. опоздание в днях
    'inst_payment_diff_mean',    # Средняя разница в сумме платежа
    'inst_paid_on_time_rate'     # Доля платежей вовремя
]

# Проверяем, какие из ожидаемых колонок реально существуют в датафрейме
# (на случай, если какой-то не создался из-за ошибки в предыдущей ячейке)
existing_hyp6_cols_to_sample = [col for col in hyp6_cols_to_sample if col in df_copy_final.columns]

# Выводим случайную выборку только для существующих колонок
if existing_hyp6_cols_to_sample:
  print(f"Случайные 5 строк для столбцов Гипотезы 6 ({', '.join(existing_hyp6_cols_to_sample)}):")
  # Применяем .sample(5) к датафрейму, выбирая только нужные колонки
  print(df_copy_final[existing_hyp6_cols_to_sample].sample(5, random_state=42)) # random_state для воспроизводимости
else:
  # Если ни одна из колонок не найдена
  print("Не найдены колонки для отображения выборки.")
  print("Убедитесь, что ячейка с кодом для Гипотезы 6 была выполнена успешно и без ошибок.")

Случайные 5 строк для столбцов Гипотезы 6 (inst_late_payment_count, inst_underpaid_count, inst_days_late_mean, inst_days_late_max, inst_payment_diff_mean, inst_paid_on_time_rate):
        inst_late_payment_count  inst_underpaid_count  inst_days_late_mean  \
245895                      3.0                   2.0            -2.882353   
98194                       0.0                   0.0           -63.052632   
36463                       0.0                   0.0             0.000000   
249923                      5.0                   2.0            -3.289474   
158389                      0.0                   0.0             0.000000   

        inst_days_late_max  inst_payment_diff_mean  inst_paid_on_time_rate  
245895                 2.0            -1587.282353                0.823529  
98194                  0.0                0.000000                1.000000  
36463                  0.0                0.000000                0.000000  
249923                 7.0             -183

## Hypothesis 7

Гипотеза: Активность использования и баланс по POS/CASH кредитам (POS_CASH_balance.csv) отражают текущее финансовое состояние.



In [18]:
# --- Реализация Гипотезы 7: Агрегация недавних данных из POS_CASH_balance ---

# 1. Добавляем sk_id_curr в POS_CASH_balance
#    (Предполагаем, что previous_application и POS_CASH_balance существуют)
prev_app_ids = previous_application[['sk_id_prev', 'sk_id_curr']]
pos_cash_merged = POS_CASH_balance.merge(prev_app_ids, on='sk_id_prev', how='left')

# 2. Обработка возможных суффиксов _x, _y для sk_id_curr (как в Гипотезе 6)
sk_id_curr_found_pc = False
if 'sk_id_curr_y' in pos_cash_merged.columns and 'sk_id_curr_x' in pos_cash_merged.columns:
    pos_cash_merged['sk_id_curr'] = pos_cash_merged['sk_id_curr_y']
    pos_cash_merged = pos_cash_merged.drop(columns=['sk_id_curr_x', 'sk_id_curr_y'])
    sk_id_curr_found_pc = True
elif 'sk_id_curr' in pos_cash_merged.columns:
    sk_id_curr_found_pc = True

# Продолжаем, только если sk_id_curr был найден/создан
if sk_id_curr_found_pc:

    # 3. Удаляем строки, где sk_id_curr остался NaN после merge
    pos_cash_merged = pos_cash_merged.dropna(subset=['sk_id_curr'])
    # pos_cash_merged['sk_id_curr'] = pos_cash_merged['sk_id_curr'].astype(int) # Опционально

    # 4. Фильтруем последние N месяцев (например, 12)
    N_MONTHS_RECENT = 12
    pos_cash_recent = pos_cash_merged[pos_cash_merged['months_balance'] >= -N_MONTHS_RECENT].copy() # Используем .copy()

    # 5. Создаем флаги на уровне месяца (в отфильтрованных данных)
    pos_cash_recent['flag_late'] = (pos_cash_recent['sk_dpd'] > 0).astype(int)
    pos_cash_recent['flag_late_def'] = (pos_cash_recent['sk_dpd_def'] > 0).astype(int)
    pos_cash_recent['flag_completed'] = (pos_cash_recent['name_contract_status'] == 'Completed').astype(int)

    # 6. Группируем по sk_id_curr и агрегируем недавнюю активность
    pos_cash_agg_recent = pos_cash_recent.groupby('sk_id_curr').agg(
        # Статистика за последние N месяцев
        pos_cash_paid_late_count_recent = ('flag_late', 'sum'),
        pos_cash_paid_late_def_count_recent = ('flag_late_def', 'sum'),
        pos_cash_avg_instalment_future_recent = ('cnt_instalment_future', 'mean'),
        pos_cash_completed_count_recent = ('flag_completed', 'sum'),
        pos_cash_recent_months_count = ('months_balance', 'count') # Кол-во записей за N мес
    ).reset_index()


    # 7. Удаляем существующие колонки из df_copy_final перед merge
    cols_to_add_pos = [col for col in pos_cash_agg_recent.columns if col != 'sk_id_curr']
    df_copy_final = df_copy_final.drop(columns=cols_to_add_pos, errors='ignore')

    # 8. Присоединяем агрегированные данные к df_copy_final
    df_copy_final = df_copy_final.merge(pos_cash_agg_recent, on='sk_id_curr', how='left')

    # 9. Обработка пропусков (NaN) для новых столбцов
    #    Заполняем нулями (если у клиента не было записей в pos_cash_recent)
    pos_cash_final_cols = [
         'pos_cash_paid_late_count_recent', 'pos_cash_paid_late_def_count_recent',
         'pos_cash_avg_instalment_future_recent', 'pos_cash_completed_count_recent',
         'pos_cash_recent_months_count'
    ]
    for col in pos_cash_final_cols:
         # Если колонки нет (из-за ошибки выше), fillna вызовет KeyError
         df_copy_final[col] = df_copy_final[col].fillna(0)

       pos_cash_paid_late_count_recent  pos_cash_paid_late_def_count_recent  \
count                    307511.000000                        307511.000000   
mean                          0.067006                             0.041277   
std                           0.566062                             0.371127   
min                           0.000000                             0.000000   
25%                           0.000000                             0.000000   
50%                           0.000000                             0.000000   
75%                           0.000000                             0.000000   
max                          30.000000                            30.000000   

       pos_cash_avg_instalment_future_recent  pos_cash_completed_count_recent  \
count                          307511.000000                    307511.000000   
mean                                7.561148                         0.540488   
std                                10.256796 

In [ ]:
cols_to_check = [col for col in pos_cash_final_cols if col in df_copy_final.columns]
if cols_to_check:
    print(df_copy_final[cols_to_check].describe())

## Hypothesis 8

Гипотеза: Использование кредитной карты (credit_card_balance.csv) и наличие задолженности по ней влияют на риск.

In [19]:
# --- Реализация Гипотезы (Кредитные карты): Агрегация данных из credit_card_balance ---

# 1. Добавляем sk_id_curr в credit_card_balance

prev_app_ids = previous_application[['sk_id_prev', 'sk_id_curr']]
cc_merged = credit_card_balance.merge(prev_app_ids, on='sk_id_prev', how='left')

# 2. Обработка возможных суффиксов _x, _y для sk_id_curr
sk_id_curr_found_cc = False
if 'sk_id_curr_y' in cc_merged.columns and 'sk_id_curr_x' in cc_merged.columns:
    cc_merged['sk_id_curr'] = cc_merged['sk_id_curr_y']
    cc_merged = cc_merged.drop(columns=['sk_id_curr_x', 'sk_id_curr_y'])
    sk_id_curr_found_cc = True
elif 'sk_id_curr' in cc_merged.columns:
    sk_id_curr_found_cc = True


# Продолжаем, только если sk_id_curr был найден/создан
if sk_id_curr_found_cc:
    cc_merged = cc_merged.dropna(subset=['sk_id_curr'])

    # 3. Создаем признак утилизации кредитного лимита на уровне месяца
    # Используем np.divide для безопасного деления (возвращает NaN при делении на 0)
    cc_merged['utilization'] = np.divide(
        cc_merged['amt_balance'],
        cc_merged['amt_credit_limit_actual']
    )
    # Заменяем inf (если лимит 0, а баланс > 0) на NaN перед агрегацией
    cc_merged['utilization'] = cc_merged['utilization'].replace([np.inf, -np.inf], np.nan)

    # 4. Группируем по sk_id_curr и агрегируем
    cc_agg = cc_merged.groupby('sk_id_curr').agg(
        cc_balance_avg = ('amt_balance', 'mean'),
        cc_balance_max = ('amt_balance', 'max'),
        cc_limit_avg = ('amt_credit_limit_actual', 'mean'),
        cc_limit_max = ('amt_credit_limit_actual', 'max'),
        cc_utilization_avg = ('utilization', 'mean'),
        cc_utilization_max = ('utilization', 'max'),
        cc_drawings_atm_avg = ('amt_drawings_atm_current', 'mean'),
        cc_drawings_atm_sum = ('amt_drawings_atm_current', 'sum'),
        cc_drawings_total_avg = ('amt_drawings_current', 'mean'),
        cc_drawings_total_sum = ('amt_drawings_current', 'sum'),
        cc_dpd_max = ('sk_dpd', 'max'),
        cc_dpd_sum = ('sk_dpd', 'sum'),
        cc_dpd_def_max = ('sk_dpd_def', 'max'),
        cc_dpd_def_sum = ('sk_dpd_def', 'sum'),
        cc_months_count = ('months_balance', 'count'),
        cc_card_count = ('sk_id_prev', 'nunique') # Считаем кол-во уникальных карт (предыдущих заявок)
    ).reset_index()


    # 5. Удаляем существующие колонки из df_copy_final перед финальным merge
    cols_to_add_cc = [col for col in cc_agg.columns if col != 'sk_id_curr']
    df_copy_final = df_copy_final.drop(columns=cols_to_add_cc, errors='ignore')

    # 6. Присоединяем агрегированные данные к df_copy_final
    df_copy_final = df_copy_final.merge(cc_agg, on='sk_id_curr', how='left')

    # 7. Обработка пропусков (NaN) для всех новых столбцов
    #    Заполняем нулями (если у клиента не было записей в credit_card_balance)
    cc_final_cols = [
        'cc_balance_avg', 'cc_balance_max', 'cc_limit_avg', 'cc_limit_max',
        'cc_utilization_avg', 'cc_utilization_max', 'cc_drawings_atm_avg',
        'cc_drawings_atm_sum', 'cc_drawings_total_avg', 'cc_drawings_total_sum',
        'cc_dpd_max', 'cc_dpd_sum', 'cc_dpd_def_max', 'cc_dpd_def_sum',
        'cc_months_count', 'cc_card_count'
    ]
    for col in cc_final_cols:
         df_copy_final[col] = df_copy_final[col].fillna(0)
         # Доп обработка для максимумов DPD
         if col in ['cc_dpd_max', 'cc_dpd_def_max']:
              df_copy_final[col] = df_copy_final[col].clip(lower=0)

       cc_balance_avg  cc_balance_max  cc_limit_avg  cc_limit_max  \
count   307511.000000    3.075110e+05  3.075110e+05  3.075110e+05   
mean     19158.982251    3.749411e+04  5.638906e+04  6.669858e+04   
std      65968.693882    1.111615e+05  1.383802e+05  1.551431e+05   
min      -2930.232558    0.000000e+00  0.000000e+00  0.000000e+00   
25%          0.000000    0.000000e+00  0.000000e+00  0.000000e+00   
50%          0.000000    0.000000e+00  0.000000e+00  0.000000e+00   
75%          0.000000    0.000000e+00  7.304899e+03  4.500000e+04   
max     928686.324286    1.354829e+06  1.350000e+06  1.350000e+06   

       cc_utilization_avg  cc_utilization_max  cc_drawings_atm_avg  \
count       307511.000000       307511.000000        307511.000000   
mean             0.081994            0.151869          2387.769847   
std              0.218811            0.358127         10884.227054   
min             -0.084848            0.000000             0.000000   
25%              0.000000   

In [ ]:
cols_to_check = [col for col in cc_final_cols if col in df_copy_final.columns]
if cols_to_check:
    print(df_copy_final[cols_to_check].describe())

## Hypothesis 9

Гипотеза: Соотношение текущей заявки и прошлой кредитной истории важно.

In [20]:
active_bureau = bureau[bureau['credit_active'] == 'Active'].copy() # Используем 'credit_active'

# 2. Группируем по ID клиента (sk_id_curr) и суммируем долг (amt_credit_sum_debt)
active_bureau_agg = active_bureau.groupby('sk_id_curr')['amt_credit_sum_debt'].sum().reset_index() # Используем 'sk_id_curr' и 'amt_credit_sum_debt'

# 3. Переименовываем колонку с суммой
active_bureau_agg.rename(columns={'amt_credit_sum_debt': 'bureau_active_debt_sum'}, inplace=True)

# 4. Присоединяем (merge) результат к основному датафрейму df_copy_final
df_copy_final = pd.merge( # <-- Используем df_copy_final
    df_copy_final,        # <-- Используем df_copy_final
    active_bureau_agg,
    on='sk_id_curr',      # Используем 'sk_id_curr'
    how='left'
)

# 5. Заполняем пропуски (NaN) нулями в датафрейме df_copy_final
df_copy_final['bureau_active_debt_sum'].fillna(0, inplace=True) # <-- Используем df_copy_final

/var/folders/vg/yb8mlk6s141fflfvctc1dpyw0000gn/T/ipykernel_6094/332713757.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_copy_final['bureau_active_debt_sum'].fillna(0, inplace=True) # <-- Используем df_copy_final


In [21]:
df_copy_final['curr_credit_over_active_bureau_debt'] = np.divide(
    df_copy_final['amt_credit'],
    df_copy_final['bureau_active_debt_sum']
)
#    Заменяем бесконечность (inf), которая возникает при делении числа > 0 на 0, на NaN
df_copy_final['curr_credit_over_active_bureau_debt'] = df_copy_final['curr_credit_over_active_bureau_debt'].replace([np.inf, -np.inf], np.nan)
#    Заполняем все NaN (возникшие от деления на ноль, деления NaN, или если исходные данные были NaN) нулем.
#    Примечание: Замена inf на 0 может быть не лучшей стратегией, иногда лучше заменить на очень большое число или оставить NaN и заполнить медианой/средним. Но для простоты пока оставим 0.
df_copy_final['curr_credit_over_active_bureau_debt'] = df_copy_final['curr_credit_over_active_bureau_debt'].fillna(0)


# 2. Отношение текущего кредита к среднему одобренному кредиту в прошлом (в HC)
#    Аналогично используем np.divide и обрабатываем inf/NaN
df_copy_final['curr_credit_over_prev_app_credit'] = np.divide(
    df_copy_final['amt_credit'],
    df_copy_final['prev_app_approved_avg_credit']
)
df_copy_final['curr_credit_over_prev_app_credit'] = df_copy_final['curr_credit_over_prev_app_credit'].replace([np.inf, -np.inf], np.nan)
df_copy_final['curr_credit_over_prev_app_credit'] = df_copy_final['curr_credit_over_prev_app_credit'].fillna(0)

In [22]:
cols_to_check_hyp9 = ['curr_credit_over_active_bureau_debt', 'curr_credit_over_prev_app_credit']
print(df_copy_final[cols_to_check_hyp9].describe())
print(df_copy_final[cols_to_check_hyp9].sample(5, random_state=42))

       curr_credit_over_active_bureau_debt  curr_credit_over_prev_app_credit
count                         3.075110e+05                     307511.000000
mean                         -6.117400e+01                          5.675792
std                           3.287252e+04                          7.170687
min                          -9.300000e+06                          0.000000
25%                           0.000000e+00                          1.600820
50%                           3.179869e-01                          3.296414
75%                           1.731251e+00                          6.970596
max                           4.672200e+06                        149.253731
        curr_credit_over_active_bureau_debt  curr_credit_over_prev_app_credit
245895                             0.000000                          0.909922
98194                              0.000000                          6.791768
36463                              3.466273                          0.00

## Hypothesis 10

Гипотеза: Тип занятости и организация могут влиять на стабильность дохода и риск.

In [23]:
# --- Реализация Гипотезы (Тип занятости и организация) ---

# --- ШАГ A: Агрегационные Признаки (Средний доход по категориям) ---

# A.1 Средний доход по типу организации
#     transform('mean') вычисляет среднее для каждой группы и возвращает Series
#     с индексом, совпадающим с df_copy_final.
#     fillna(0) обрабатывает случаи, если для какой-то организации нет данных о доходе
#     или появляются новые организации (например, в тестовом наборе).
df_copy_final['income_mean_by_org'] = df_copy_final.groupby('organization_type')['amt_income_total'].transform('mean')
df_copy_final['income_mean_by_org'] = df_copy_final['income_mean_by_org'].fillna(0)

# A.2 Средний доход по типу занятости
#     transform проигнорирует строки с NaN в 'occupation_type' при группировке.
#     fillna(0) обработает и эти строки, и те профессии, где не было данных о доходе.
df_copy_final['income_mean_by_occ'] = df_copy_final.groupby('occupation_type')['amt_income_total'].transform('mean')
df_copy_final['income_mean_by_occ'] = df_copy_final['income_mean_by_occ'].fillna(0)

# --- ШАГ B: Кодирование Категориальных Признаков ---

# B.1 Обработка 'occupation_type' -> One-Hot Encoding
occupation_nan_filler = 'XNA' # Задаем значение для заполнения пропусков
# Сначала заполняем пропуски в исходной колонке
df_copy_final['occupation_type'] = df_copy_final['occupation_type'].fillna(occupation_nan_filler)
# Затем применяем OHE. Исходный столбец 'occupation_type' будет удален
# и заменен на новые столбцы вида 'occ_Accountants', 'occ_Laborers' и т.д.
df_copy_final = pd.get_dummies(df_copy_final, columns=['occupation_type'], prefix='occ', prefix_sep='_')

# B.2 Обработка 'organization_type' -> Frequency Encoding
#     Считаем, сколько раз встречается каждая организация
org_type_counts = df_copy_final['organization_type'].value_counts()
#     Создаем новую колонку, где вместо названия организации будет ее частота
df_copy_final['org_type_freq_encoded'] = df_copy_final['organization_type'].map(org_type_counts)
#     Заполняем NaN нулем (на случай появления новых организаций в тесте)
df_copy_final['org_type_freq_encoded'] = df_copy_final['org_type_freq_encoded'].fillna(0)

In [24]:
# Находим все новые OHE колонки (начинаются с 'occ_')
ohe_cols_hyp10 = [col for col in df_copy_final.columns if col.startswith('occ_')]

if ohe_cols_hyp10:
    print(f"\n--- Проверка One-Hot Encoded признаков (профессия) ---")
    print(f"Найдено {len(ohe_cols_hyp10)} OHE колонок.")
    print("Примеры названий:", ohe_cols_hyp10[:5]) # Показываем первые 5

    # Выведем несколько OHE колонок для случайных 5 строк
    cols_to_show_ohe = ohe_cols_hyp10[:4] # Берем первые 4 для примера
    print("\nПример значений OHE (случайные 5 строк, первые неск. колонок):")
    if 'sk_id_curr' in df_copy_final.columns:
        cols_to_show_ohe = ['sk_id_curr'] + cols_to_show_ohe # Добавляем ID для контекста
    print(df_copy_final.loc[df_copy_final.sample(5, random_state=42).index, cols_to_show_ohe])


    # Проверим суммы по OHE колонкам (сколько единиц в каждой)
    print("\nСуммы по первым 5 OHE колонкам (кол-во клиентов с этой профессией):")
    print(df_copy_final[ohe_cols_hyp10[:5]].sum())

    # Проверим, удалена ли исходная колонка 'occupation_type'
    print(f"\n'occupation_type' в колонках: {'occupation_type' in df_copy_final.columns}") # Должно быть False
else:
    print("Не найдены OHE колонки (с префиксом 'occ_').")


--- Проверка One-Hot Encoded признаков (профессия) ---
Найдено 19 OHE колонок.
Примеры названий: ['occ_', 'occ_Accountants', 'occ_Cleaning staff', 'occ_Cooking staff', 'occ_Core staff']

Пример значений OHE (случайные 5 строк, первые неск. колонок):
        sk_id_curr   occ_  occ_Accountants  occ_Cleaning staff  \
245895      147610   True            False               False   
98194       332891  False            False               False   
36463       261387  False            False               False   
249923      152294   True            False               False   
158389      402194   True            False               False   

        occ_Cooking staff  
245895              False  
98194               False  
36463               False  
249923              False  
158389              False  

Суммы по первым 5 OHE колонкам (кол-во клиентов с этой профессией):
occ_                  96391
occ_Accountants        9813
occ_Cleaning staff     4653
occ_Cooking staff      5946
occ

In [30]:
oof_df, cv_metrics, final_models = run_training_pipeline(df=df_copy_final)

--- Запуск Пайплайна Обучения (с возвратом моделей) ---
!!! ВНИМАНИЕ: Настройки 2/2 для быстрой проверки, НЕ для надежной оценки/тюнинга !!!
Автоматическое определение списка признаков...
  Используется 152 признаков (автоматически определено).

2. Подготовка данных...
  Размер данных для обучения: (307511, 152)

3. Настройка MLflow...


2025/04/21 19:39:06 INFO mlflow.tracking.fluent: Experiment with name 'Pipeline Run Returning Models (2:2 Example)' does not exist. Creating a new experiment.


  Эксперимент MLflow: Pipeline Run Returning Models (2:2 Example) на http://82.202.137.136:8000
  Используется стратегия: 2 фолда CV.

4. Запуск основного цикла HPO и CV (2 фолда CV)...

  --- Обработка модели: LightGBM ---


[I 2025-04-21 19:39:09,983] A new study created in memory with name: LightGBM Optuna for run 06ac07a440724b159529b3aa8efe7005


    Подбор гиперпараметров (2 итерации, 2 фолда CV)...


...

    Лучший AUC (Optuna, 2-Fold CV): 0.77436
    Кросс-валидация (2 фолдов) с лучшими параметрами...
    Средние метрики CV (2 фолда) для XGBoost:
      Auc: 0.77436 +/- 0.00049
      Logloss: 0.24142 +/- 0.00008
      Accuracy: 0.91965 +/- 0.00012
      Precision: 0.53029 +/- 0.00996
      Recall: 0.04064 +/- 0.00044
      F1: 0.07550 +/- 0.00086
    Время обработки XGBoost: 165.88 сек.

  --- Обработка модели: CatBoost ---


[I 2025-04-21 19:44:32,480] A new study created in memory with name: CatBoost Optuna for run 06ac07a440724b159529b3aa8efe7005


    Подбор гиперпараметров (2 итерации, 2 фолда CV)...


[I 2025-04-21 19:48:50,425] Trial 0 finished with value: 0.7772643241858838 and parameters: {'learning_rate': 0.013412775060807836, 'depth': 7, 'l2_leaf_reg': 2.2781742650802053, 'border_count': 253, 'subsample': 0.9198956970030485}. Best is trial 0 with value: 0.7772643241858838.
[I 2025-04-21 19:52:27,610] Trial 1 finished with value: 0.7697599182893748 and parameters: {'learning_rate': 0.030513489145430693, 'depth': 10, 'l2_leaf_reg': 0.625393135836193, 'border_count': 152, 'subsample': 0.5801662801145968}. Best is trial 0 with value: 0.7772643241858838.


    Лучший AUC (Optuna, 2-Fold CV): 0.77726
    Кросс-валидация (2 фолдов) с лучшими параметрами...
    Средние метрики CV (2 фолда) для CatBoost:
      Auc: 0.77726 +/- 0.00004
      Logloss: 0.24032 +/- 0.00006
      Accuracy: 0.91966 +/- 0.00012
      Precision: 0.54409 +/- 0.01548
      Recall: 0.03057 +/- 0.00069
      F1: 0.05789 +/- 0.00114
    Время обработки CatBoost: 818.67 сек.
🏃 View run Model Comparison Run 2:2 at: http://82.202.137.136:8000/#/experiments/819865738036629451/runs/06ac07a440724b159529b3aa8efe7005
🧪 View experiment at: http://82.202.137.136:8000/#/experiments/819865738036629451

5. Обучение финальных моделей на всех данных X, y...
  Обучение финальной модели: LightGBM
  Модель LightGBM обучена.
  Обучение финальной модели: XGBoost
  Модель XGBoost обучена.
  Обучение финальной модели: CatBoost
  Модель CatBoost обучена.

--- Пайплайн завершен ---


In [34]:
# Теперь у вас есть обученные модели:
final_model_lgbm = final_models['LightGBM']
final_model_xgb = final_models['XGBoost']
final_model_cb = final_models['CatBoost']

print("Финальные модели получены!")

Финальные модели получены!


In [25]:
training_df = df_copy_final.copy()

# Submission

In [26]:
df_copy_final = df_test.copy()

In [27]:
df_copy_final['credit_income_ratio'] = df_copy_final['amt_credit'] / df_copy_final['amt_income_total']

# 2. Создаем столбец annuity_income_ratio
df_copy_final['annuity_income_ratio'] = df_copy_final['amt_annuity'] / df_copy_final['amt_income_total']

# 3. Создаем столбец credit_term
df_copy_final['credit_term'] = df_copy_final['amt_credit'] / df_copy_final['amt_annuity']
# 1. Создаем столбец 'age' (возраст в годах)
# Делим на -365, так как days_birth отрицательный
df_copy_final['age'] = df_copy_final['days_birth'] / -365

# 2. Создаем столбец 'years_employed' (стаж в годах)
# Делим на 365, так как очищенный 'days_employed' уже положительный (или NaN)
df_copy_final['years_employed'] = df_copy_final['days_employed'] / 365

# 3. Создаем столбец 'employed_birth_ratio' (доля рабочей жизни)
# Используем модуль от days_birth и очищенный days_employed
# abs() берет модуль от отрицательного days_birth
df_copy_final['employed_birth_ratio'] = df_copy_final['days_employed'] / df_copy_final['days_birth'].abs()

# 4. Создаем полиномиальные признаки (квадраты)
# Возводим в квадрат возраст и стаж
df_copy_final['age_sq'] = df_copy_final['age'] ** 2
df_copy_final['years_employed_sq'] = df_copy_final['years_employed'] ** 2 # NaN при возведении в квадрат останется NaN

# 5. Создаем признак взаимодействия
# Перемножаем возраст и стаж
df_copy_final['age_emp_interaction'] = df_copy_final['age'] * df_copy_final['years_employed'] # NaN при умножении останется NaN
df_copy_final['flag_own_car'] = (df_copy_final['flag_own_car'] == 'Y').astype(int)
df_copy_final['flag_own_realty'] = (df_copy_final['flag_own_realty'] == 'Y').astype(int)

# 2. Создаем признак взаимодействия 'owns_car_and_realty'
#    Выполнение дойдет сюда, только если оба столбца существовали на шаге 1.
df_copy_final['owns_car_and_realty'] = df_copy_final['flag_own_car'] * df_copy_final['flag_own_realty']
bureau_agg_general = bureau.groupby('sk_id_curr').agg(
    bureau_loan_count = ('sk_id_bureau', 'count'),
    bureau_loan_types_count = ('credit_type', 'nunique'),
    bureau_avg_prolong_count = ('cnt_credit_prolong', 'mean'),
    bureau_total_debt_sum = ('amt_credit_sum_debt', 'sum'),
    bureau_max_days_overdue = ('credit_day_overdue', 'max'),
    bureau_overdue_loan_count = ('amt_credit_sum_overdue', lambda x: (x > 0).sum())
).reset_index()

# Список колонок, которые мы создали в bureau_agg_general (кроме ключа 'sk_id_curr')
cols_to_add = [col for col in bureau_agg_general.columns if col != 'sk_id_curr']

# Находим, какие из этих колонок УЖЕ ЕСТЬ в df_copy_final
cols_to_drop = [col for col in cols_to_add if col in df_copy_final.columns]

# Если такие колонки нашлись, удаляем их из df_copy_final
if cols_to_drop:
    # print(f"Удаляем существующие столбцы перед merge: {cols_to_drop}") # Можно раскомментировать для информации
    df_copy_final = df_copy_final.drop(columns=cols_to_drop)
# -----------------------------------------------------------------

# 2. Присоединяем (merge) агрегированные данные к df_copy_final
df_copy_final = df_copy_final.merge(bureau_agg_general, on='sk_id_curr', how='left')

# 3. Обработка пропусков (NaN) после мержа
bureau_general_agg_cols = [
    'bureau_loan_count', 'bureau_loan_types_count', 'bureau_avg_prolong_count',
    'bureau_total_debt_sum', 'bureau_max_days_overdue', 'bureau_overdue_loan_count'
]
for col in bureau_general_agg_cols:
     if col in df_copy_final.columns:
        df_copy_final[col] = df_copy_final[col].fillna(0)
        if col == 'bureau_max_days_overdue':
             df_copy_final[col] = df_copy_final[col].clip(lower=0)

prev_app_agg = previous_application.groupby('sk_id_curr').agg(
    # Общее количество и количество по статусам
    prev_app_count = ('sk_id_prev', 'count'),
    prev_app_approved_count = ('name_contract_status', lambda x: (x == 'Approved').sum()),
    prev_app_refused_count = ('name_contract_status', lambda x: (x == 'Refused').sum()),

    # Рассчитываем СУММЫ по одобренным заявкам, чтобы позже найти среднее.
    # Используем .loc для корректного выбора строк внутри лямбды по индексу группы x.index
    prev_app_approved_sum_credit = ('amt_credit', lambda x: previous_application.loc[x.index, 'amt_credit'][previous_application.loc[x.index, 'name_contract_status'] == 'Approved'].sum()),
    prev_app_approved_sum_annuity = ('amt_annuity', lambda x: previous_application.loc[x.index, 'amt_annuity'][previous_application.loc[x.index, 'name_contract_status'] == 'Approved'].sum())

).reset_index() # Возвращаем sk_id_curr из индекса в столбец

# 2. Удаляем существующие колонки из df_copy_final перед merge (если они есть)
#    Это предотвратит MergeError при повторном запуске ячейки
cols_to_add_prev = [col for col in prev_app_agg.columns if col != 'sk_id_curr']
cols_to_drop_prev = [col for col in cols_to_add_prev if col in df_copy_final.columns]
if cols_to_drop_prev:
    df_copy_final = df_copy_final.drop(columns=cols_to_drop_prev)

# 3. Присоединяем (merge) агрегированные данные к df_copy_final
df_copy_final = df_copy_final.merge(prev_app_agg, on='sk_id_curr', how='left')

# 4. Вычисляем производные признаки (доля и средние) ПОСЛЕ merge
#    Это безопаснее, т.к. мы можем обработать деление на ноль (если prev_app_count или prev_app_approved_count равны 0)

# Доля одобренных заявок
# np.divide обрабатывает деление на ноль, можно заменить результат (inf) на NaN, затем на 0
df_copy_final['prev_app_approved_rate'] = (
    df_copy_final['prev_app_approved_count'] / df_copy_final['prev_app_count']
)
# Заменяем inf (деление на 0) и -inf на NaN, затем все NaN на 0
df_copy_final['prev_app_approved_rate'] = df_copy_final['prev_app_approved_rate'].replace([np.inf, -np.inf], np.nan).fillna(0)


# Средняя сумма кредита по одобренным
df_copy_final['prev_app_approved_avg_credit'] = (
    df_copy_final['prev_app_approved_sum_credit'] / df_copy_final['prev_app_approved_count']
)
df_copy_final['prev_app_approved_avg_credit'] = df_copy_final['prev_app_approved_avg_credit'].replace([np.inf, -np.inf], np.nan).fillna(0)


# Средняя сумма аннуитета по одобренным
df_copy_final['prev_app_approved_avg_annuity'] = (
    df_copy_final['prev_app_approved_sum_annuity'] / df_copy_final['prev_app_approved_count']
)
df_copy_final['prev_app_approved_avg_annuity'] = df_copy_final['prev_app_approved_avg_annuity'].replace([np.inf, -np.inf], np.nan).fillna(0)


# 5. Обработка пропусков (NaN) для всех добавленных/вычисленных столбцов
#    Заполняем нулями столбцы, которые пришли из merge (если sk_id_curr не было в previous_application)
#    или были вычислены (если были NaN после деления)
prev_app_final_cols = [
    'prev_app_count', 'prev_app_approved_count', 'prev_app_refused_count',
    'prev_app_approved_sum_credit', 'prev_app_approved_sum_annuity', # Промежуточные суммы
    'prev_app_approved_rate', 'prev_app_approved_avg_credit', 'prev_app_approved_avg_annuity' # Финальные признаки
]
for col in prev_app_final_cols:
     if col in df_copy_final.columns:
        df_copy_final[col] = df_copy_final[col].fillna(0)

# --- Реализация Гипотезы 6 (Минимальная версия, без импортов) ---

# 1. Добавляем sk_id_curr в installments_payments
prev_app_ids = previous_application[['sk_id_prev', 'sk_id_curr']]
inst_merged = installments_payments.merge(prev_app_ids, on='sk_id_prev', how='left')
# del prev_app_ids # Опционально

# 2. Обработка возможных суффиксов _x, _y для sk_id_curr (НЕОБХОДИМОЕ УСЛОВИЕ)
if 'sk_id_curr_y' in inst_merged.columns and 'sk_id_curr_x' in inst_merged.columns:
    inst_merged['sk_id_curr'] = inst_merged['sk_id_curr_y']
    inst_merged = inst_merged.drop(columns=['sk_id_curr_x', 'sk_id_curr_y'])
# Если колонки sk_id_curr нет (ни с суффиксами, ни без), последующий код вызовет KeyError или NameError

# 3. Удаляем строки, где sk_id_curr остался NaN после merge
#    Предполагаем, что sk_id_curr теперь существует, иначе будет KeyError
inst_merged = inst_merged.dropna(subset=['sk_id_curr'])


# 4. Создаем признаки на уровне каждого платежа
inst_merged['payment_diff'] = inst_merged['amt_payment'] - inst_merged['amt_instalment']
inst_merged['days_late'] = inst_merged['days_entry_payment'] - inst_merged['days_instalment']
inst_merged['paid_late_flag'] = (inst_merged['days_late'] > 0).astype(int)
inst_merged['underpaid_flag'] = (inst_merged['payment_diff'] < -0.001).astype(int)
inst_merged['paid_on_time_flag'] = (inst_merged['days_late'] <= 0).astype(int)

# 5. Группируем по sk_id_curr и агрегируем
#    Предполагаем, что все исходные колонки существуют, иначе будет KeyError
inst_agg = inst_merged.groupby('sk_id_curr').agg(
    inst_payment_diff_mean = ('payment_diff', 'mean'),
    inst_payment_diff_max = ('payment_diff', 'max'),
    inst_payment_diff_sum = ('payment_diff', 'sum'),
    inst_days_late_mean = ('days_late', 'mean'),
    inst_days_late_max = ('days_late', 'max'),
    inst_days_late_sum = ('days_late', 'sum'),
    inst_late_payment_count = ('paid_late_flag', 'sum'),
    inst_underpaid_count = ('underpaid_flag', 'sum'),
    inst_paid_on_time_count = ('paid_on_time_flag', 'sum'),
    inst_total_payment_count = ('sk_id_prev', 'count')
).reset_index()


# 6. Удаляем существующие колонки из df_copy_final перед финальным merge
#    Используем errors='ignore' чтобы не проверять наличие колонок явно
cols_to_add_inst = [col for col in inst_agg.columns if col != 'sk_id_curr']
df_copy_final = df_copy_final.drop(columns=cols_to_add_inst, errors='ignore')

# 7. Присоединяем агрегированные данные к df_copy_final
df_copy_final = df_copy_final.merge(inst_agg, on='sk_id_curr', how='left')

# 8. Вычисляем долю платежей вовремя ПОСЛЕ merge
df_copy_final['inst_paid_on_time_rate'] = np.divide(
    df_copy_final['inst_paid_on_time_count'],
    df_copy_final['inst_total_payment_count']
)

# 9. Обработка пропусков (NaN) для всех новых столбцов
#    Предполагаем, что все эти колонки были успешно добавлены/созданы
#    Если какой-то колонки нет, fillna вызовет ошибку KeyError
inst_final_cols = [
    'inst_payment_diff_mean', 'inst_payment_diff_max', 'inst_payment_diff_sum',
    'inst_days_late_mean', 'inst_days_late_max', 'inst_days_late_sum',
    'inst_late_payment_count', 'inst_underpaid_count',
    'inst_paid_on_time_count', 'inst_total_payment_count',
    'inst_paid_on_time_rate'
]
for col in inst_final_cols:
     df_copy_final[col] = df_copy_final[col].fillna(0)
     if col == 'inst_days_late_max':
          df_copy_final[col] = df_copy_final[col].clip(lower=0)

# --- Реализация Гипотезы 7: Агрегация недавних данных из POS_CASH_balance ---

# 1. Добавляем sk_id_curr в POS_CASH_balance
#    (Предполагаем, что previous_application и POS_CASH_balance существуют)
prev_app_ids = previous_application[['sk_id_prev', 'sk_id_curr']]
pos_cash_merged = POS_CASH_balance.merge(prev_app_ids, on='sk_id_prev', how='left')


# 2. Обработка возможных суффиксов _x, _y для sk_id_curr (как в Гипотезе 6)
sk_id_curr_found_pc = False
if 'sk_id_curr_y' in pos_cash_merged.columns and 'sk_id_curr_x' in pos_cash_merged.columns:
    pos_cash_merged['sk_id_curr'] = pos_cash_merged['sk_id_curr_y']
    pos_cash_merged = pos_cash_merged.drop(columns=['sk_id_curr_x', 'sk_id_curr_y'])
    sk_id_curr_found_pc = True
elif 'sk_id_curr' in pos_cash_merged.columns:
    sk_id_curr_found_pc = True


# Продолжаем, только если sk_id_curr был найден/создан
if sk_id_curr_found_pc:

    # 3. Удаляем строки, где sk_id_curr остался NaN после merge
    pos_cash_merged = pos_cash_merged.dropna(subset=['sk_id_curr'])
    # pos_cash_merged['sk_id_curr'] = pos_cash_merged['sk_id_curr'].astype(int) # Опционально

    # 4. Фильтруем последние N месяцев (например, 12)
    N_MONTHS_RECENT = 12
    pos_cash_recent = pos_cash_merged[pos_cash_merged['months_balance'] >= -N_MONTHS_RECENT].copy() # Используем .copy()

    # 5. Создаем флаги на уровне месяца (в отфильтрованных данных)
    pos_cash_recent['flag_late'] = (pos_cash_recent['sk_dpd'] > 0).astype(int)
    pos_cash_recent['flag_late_def'] = (pos_cash_recent['sk_dpd_def'] > 0).astype(int)
    pos_cash_recent['flag_completed'] = (pos_cash_recent['name_contract_status'] == 'Completed').astype(int)

    # 6. Группируем по sk_id_curr и агрегируем недавнюю активность
    pos_cash_agg_recent = pos_cash_recent.groupby('sk_id_curr').agg(
        # Статистика за последние N месяцев
        pos_cash_paid_late_count_recent = ('flag_late', 'sum'),
        pos_cash_paid_late_def_count_recent = ('flag_late_def', 'sum'),
        pos_cash_avg_instalment_future_recent = ('cnt_instalment_future', 'mean'),
        pos_cash_completed_count_recent = ('flag_completed', 'sum'),
        pos_cash_recent_months_count = ('months_balance', 'count') # Кол-во записей за N мес
    ).reset_index()

    # del pos_cash_merged, pos_cash_recent # Опционально

    # 7. Удаляем существующие колонки из df_copy_final перед merge
    cols_to_add_pos = [col for col in pos_cash_agg_recent.columns if col != 'sk_id_curr']
    df_copy_final = df_copy_final.drop(columns=cols_to_add_pos, errors='ignore')

    # 8. Присоединяем агрегированные данные к df_copy_final
    df_copy_final = df_copy_final.merge(pos_cash_agg_recent, on='sk_id_curr', how='left')

    # 9. Обработка пропусков (NaN) для новых столбцов
    #    Заполняем нулями (если у клиента не было записей в pos_cash_recent)
    pos_cash_final_cols = [
         'pos_cash_paid_late_count_recent', 'pos_cash_paid_late_def_count_recent',
         'pos_cash_avg_instalment_future_recent', 'pos_cash_completed_count_recent',
         'pos_cash_recent_months_count'
    ]
    for col in pos_cash_final_cols:
         # Если колонки нет (из-за ошибки выше), fillna вызовет KeyError
         df_copy_final[col] = df_copy_final[col].fillna(0)

cols_to_check = [col for col in pos_cash_final_cols if col in df_copy_final.columns]
if cols_to_check: 
    print(df_copy_final[cols_to_check].describe())

# --- Реализация Гипотезы (Кредитные карты): Агрегация данных из credit_card_balance ---

# 1. Добавляем sk_id_curr в credit_card_balance
#    (Предполагаем, что previous_application и credit_card_balance существуют)
prev_app_ids = previous_application[['sk_id_prev', 'sk_id_curr']]
cc_merged = credit_card_balance.merge(prev_app_ids, on='sk_id_prev', how='left')
# del prev_app_ids # Опционально

# 2. Обработка возможных суффиксов _x, _y для sk_id_curr
sk_id_curr_found_cc = False
if 'sk_id_curr_y' in cc_merged.columns and 'sk_id_curr_x' in cc_merged.columns:
    cc_merged['sk_id_curr'] = cc_merged['sk_id_curr_y']
    cc_merged = cc_merged.drop(columns=['sk_id_curr_x', 'sk_id_curr_y'])
    sk_id_curr_found_cc = True
elif 'sk_id_curr' in cc_merged.columns:
    sk_id_curr_found_cc = True


# Продолжаем, только если sk_id_curr был найден/создан
if sk_id_curr_found_cc:
    cc_merged = cc_merged.dropna(subset=['sk_id_curr'])
    

    # 3. Создаем признак утилизации кредитного лимита на уровне месяца
    # Используем np.divide для безопасного деления (возвращает NaN при делении на 0)
    cc_merged['utilization'] = np.divide(
        cc_merged['amt_balance'],
        cc_merged['amt_credit_limit_actual']
    )
    # Заменяем inf (если лимит 0, а баланс > 0) на NaN перед агрегацией
    cc_merged['utilization'] = cc_merged['utilization'].replace([np.inf, -np.inf], np.nan)

    # 4. Группируем по sk_id_curr и агрегируем
    cc_agg = cc_merged.groupby('sk_id_curr').agg(
        cc_balance_avg = ('amt_balance', 'mean'),
        cc_balance_max = ('amt_balance', 'max'),
        cc_limit_avg = ('amt_credit_limit_actual', 'mean'),
        cc_limit_max = ('amt_credit_limit_actual', 'max'),
        cc_utilization_avg = ('utilization', 'mean'), # NaNы (от деления на 0 или исходные) будут проигнорированы mean/max
        cc_utilization_max = ('utilization', 'max'),
        cc_drawings_atm_avg = ('amt_drawings_atm_current', 'mean'),
        cc_drawings_atm_sum = ('amt_drawings_atm_current', 'sum'),
        cc_drawings_total_avg = ('amt_drawings_current', 'mean'),
        cc_drawings_total_sum = ('amt_drawings_current', 'sum'),
        cc_dpd_max = ('sk_dpd', 'max'),
        cc_dpd_sum = ('sk_dpd', 'sum'),
        cc_dpd_def_max = ('sk_dpd_def', 'max'),
        cc_dpd_def_sum = ('sk_dpd_def', 'sum'),
        cc_months_count = ('months_balance', 'count'),
        cc_card_count = ('sk_id_prev', 'nunique') # Считаем кол-во уникальных карт (предыдущих заявок)
    ).reset_index()



    # 5. Удаляем существующие колонки из df_copy_final перед финальным merge
    cols_to_add_cc = [col for col in cc_agg.columns if col != 'sk_id_curr']
    df_copy_final = df_copy_final.drop(columns=cols_to_add_cc, errors='ignore')

    # 6. Присоединяем агрегированные данные к df_copy_final
    df_copy_final = df_copy_final.merge(cc_agg, on='sk_id_curr', how='left')

    # 7. Обработка пропусков (NaN) для всех новых столбцов
    #    Заполняем нулями (если у клиента не было записей в credit_card_balance)
    cc_final_cols = [
        'cc_balance_avg', 'cc_balance_max', 'cc_limit_avg', 'cc_limit_max',
        'cc_utilization_avg', 'cc_utilization_max', 'cc_drawings_atm_avg',
        'cc_drawings_atm_sum', 'cc_drawings_total_avg', 'cc_drawings_total_sum',
        'cc_dpd_max', 'cc_dpd_sum', 'cc_dpd_def_max', 'cc_dpd_def_sum',
        'cc_months_count', 'cc_card_count'
    ]
    for col in cc_final_cols:
         # Если колонки col нет в df_copy_final (из-за ошибки выше), fillna вызовет KeyError
         df_copy_final[col] = df_copy_final[col].fillna(0)
         # Доп обработка для максимумов DPD
         if col in ['cc_dpd_max', 'cc_dpd_def_max']:
              df_copy_final[col] = df_copy_final[col].clip(lower=0)

    

cols_to_check = [col for col in cc_final_cols if col in df_copy_final.columns]
if cols_to_check:
    print(df_copy_final[cols_to_check].describe())
active_bureau = bureau[bureau['credit_active'] == 'Active'].copy() # Используем 'credit_active'

# 2. Группируем по ID клиента (sk_id_curr) и суммируем долг (amt_credit_sum_debt)
active_bureau_agg = active_bureau.groupby('sk_id_curr')['amt_credit_sum_debt'].sum().reset_index() # Используем 'sk_id_curr' и 'amt_credit_sum_debt'

# 3. Переименовываем колонку с суммой
active_bureau_agg.rename(columns={'amt_credit_sum_debt': 'bureau_active_debt_sum'}, inplace=True)

# 4. Присоединяем (merge) результат к основному датафрейму df_copy_final
df_copy_final = pd.merge( # <-- Используем df_copy_final
    df_copy_final,        # <-- Используем df_copy_final
    active_bureau_agg,
    on='sk_id_curr',      # Используем 'sk_id_curr'
    how='left'
)

# 5. Заполняем пропуски (NaN) нулями в датафрейме df_copy_final
df_copy_final['bureau_active_debt_sum'].fillna(0, inplace=True) # <-- Используем df_copy_final


df_copy_final['curr_credit_over_active_bureau_debt'] = np.divide(
    df_copy_final['amt_credit'],
    df_copy_final['bureau_active_debt_sum']
)
#    Заменяем бесконечность (inf), которая возникает при делении числа > 0 на 0, на NaN
df_copy_final['curr_credit_over_active_bureau_debt'] = df_copy_final['curr_credit_over_active_bureau_debt'].replace([np.inf, -np.inf], np.nan)
df_copy_final['curr_credit_over_active_bureau_debt'] = df_copy_final['curr_credit_over_active_bureau_debt'].fillna(0)


# 2. Отношение текущего кредита к среднему одобренному кредиту в прошлом (в HC)
#    Аналогично используем np.divide и обрабатываем inf/NaN
df_copy_final['curr_credit_over_prev_app_credit'] = np.divide(
    df_copy_final['amt_credit'],
    df_copy_final['prev_app_approved_avg_credit']
)
df_copy_final['curr_credit_over_prev_app_credit'] = df_copy_final['curr_credit_over_prev_app_credit'].replace([np.inf, -np.inf], np.nan)
df_copy_final['curr_credit_over_prev_app_credit'] = df_copy_final['curr_credit_over_prev_app_credit'].fillna(0)


# --- Реализация Гипотезы (Тип занятости и организация) ---

# --- ШАГ A: Агрегационные Признаки (Средний доход по категориям) ---

# A.1 Средний доход по типу организации
#     transform('mean') вычисляет среднее для каждой группы и возвращает Series
#     с индексом, совпадающим с df_copy_final.
#     fillna(0) обрабатывает случаи, если для какой-то организации нет данных о доходе
#     или появляются новые организации (например, в тестовом наборе).
df_copy_final['income_mean_by_org'] = df_copy_final.groupby('organization_type')['amt_income_total'].transform('mean')
df_copy_final['income_mean_by_org'] = df_copy_final['income_mean_by_org'].fillna(0)

# A.2 Средний доход по типу занятости
#     transform проигнорирует строки с NaN в 'occupation_type' при группировке.
#     fillna(0) обработает и эти строки, и те профессии, где не было данных о доходе.
df_copy_final['income_mean_by_occ'] = df_copy_final.groupby('occupation_type')['amt_income_total'].transform('mean')
df_copy_final['income_mean_by_occ'] = df_copy_final['income_mean_by_occ'].fillna(0)

# --- ШАГ B: Кодирование Категориальных Признаков ---

# B.1 Обработка 'occupation_type' -> One-Hot Encoding
occupation_nan_filler = 'XNA' # Задаем значение для заполнения пропусков
# Сначала заполняем пропуски в исходной колонке
df_copy_final['occupation_type'] = df_copy_final['occupation_type'].fillna(occupation_nan_filler)
# Затем применяем OHE. Исходный столбец 'occupation_type' будет удален
# и заменен на новые столбцы вида 'occ_Accountants', 'occ_Laborers' и т.д.
df_copy_final = pd.get_dummies(df_copy_final, columns=['occupation_type'], prefix='occ', prefix_sep='_')

# B.2 Обработка 'organization_type' -> Frequency Encoding
#     Считаем, сколько раз встречается каждая организация
org_type_counts = df_copy_final['organization_type'].value_counts()
#     Создаем новую колонку, где вместо названия организации будет ее частота
df_copy_final['org_type_freq_encoded'] = df_copy_final['organization_type'].map(org_type_counts)
#     Заполняем NaN нулем (на случай появления новых организаций в тесте)
df_copy_final['org_type_freq_encoded'] = df_copy_final['org_type_freq_encoded'].fillna(0)


       pos_cash_paid_late_count_recent  pos_cash_paid_late_def_count_recent  \
count                     48744.000000                         48744.000000   
mean                          0.059105                             0.034507   
std                           0.463658                             0.278328   
min                           0.000000                             0.000000   
25%                           0.000000                             0.000000   
50%                           0.000000                             0.000000   
75%                           0.000000                             0.000000   
max                          12.000000                             8.000000   

       pos_cash_avg_instalment_future_recent  pos_cash_completed_count_recent  \
count                           48744.000000                     48744.000000   
mean                                6.932262                         0.525706   
std                                 9.464003 

/var/folders/vg/yb8mlk6s141fflfvctc1dpyw0000gn/T/ipykernel_6094/2770464009.py:412: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_copy_final['bureau_active_debt_sum'].fillna(0, inplace=True) # <-- Используем df_copy_final


In [29]:
preprocessor = Preprocessing()
preprocessor.fit(training_df)
train_processed = preprocessor.transform(training_df)
test_processed = preprocessor.transform(df_copy_final)

2025-04-22 17:21:51,755 - INFO - Starting preprocessing fitting...
2025-04-22 17:21:52,353 - INFO - Target column 'target' excluded for fitting.
2025-04-22 17:21:52,571 - INFO - Identified 169 numerical feature columns.
2025-04-22 17:21:52,571 - INFO - Identified 13 categorical feature columns.
2025-04-22 17:21:56,828 - INFO - Numerical imputer fitted.
2025-04-22 17:21:57,442 - INFO - Categorical imputer fitted.
2025-04-22 17:21:57,796 - INFO - Generated 123 raw OHE feature names.
2025-04-22 17:21:57,797 - INFO - Sanitized OHE feature names. Count: 123
2025-04-22 17:21:57,797 - INFO - Preprocessing fitting finished.
2025-04-22 17:21:57,878 - INFO - Starting preprocessing transform for dataframe with shape (307511, 202)...
2025-04-22 17:21:58,658 - INFO - Numerical imputation applied.
2025-04-22 17:21:58,986 - INFO - Categorical imputation applied.
/Users/daniilmerkulov/Documents/codding/mentorship/kaggle-competition-task/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:27

In [30]:
oof_results, cv_metrics, final_models = run_training_pipeline(df=train_processed)

2025-04-22 17:22:19,051 - INFO - --- Запуск Пайплайна Обучения (с возвратом моделей) ---
2025-04-22 17:22:19,055 - WARNING - !!! ВНИМАНИЕ: Настройки N_SPLITS=2, N_TRIALS_OPTUNA=2 для быстрой проверки, НЕ для надежной оценки/тюнинга !!!
2025-04-22 17:22:19,056 - INFO - Определение списка признаков из предобработанного датафрейма...
2025-04-22 17:22:19,057 - INFO -   Используется 310 признаков.
2025-04-22 17:22:19,057 - INFO - 2. Подготовка данных...
2025-04-22 17:22:19,845 - INFO -   Размер данных для обучения (X, y): (307511, 310), (307511,)
2025-04-22 17:22:20,231 - INFO - 3. Настройка MLflow...
2025-04-22 17:22:20,583 - INFO -   Эксперимент MLflow: Pipeline Run Returning Models (2:2 Example) на http://82.202.137.136:8000
2025-04-22 17:22:20,585 - INFO -   Используется стратегия: 2 фолда StratifiedKFold.
2025-04-22 17:22:20,587 - INFO - 4. Запуск основного цикла HPO и CV (2 фолда CV)...
2025-04-22 17:22:21,047 - INFO - MLflow Parent Run ID: c6ff2af256044637b89b265b6baf79de
2025-04-22 

🏃 View run LightGBM HPO+CV at: http://82.202.137.136:8000/#/experiments/819865738036629451/runs/50daa75535b948fd9dc290e0b9c8a0a5
🧪 View experiment at: http://82.202.137.136:8000/#/experiments/819865738036629451


2025-04-22 17:25:30,812 - INFO -   Время обработки LightGBM: 188.17 сек.
2025-04-22 17:25:31,133 - INFO - --- Обработка модели: XGBoost ---
2025-04-22 17:25:31,489 - INFO -   MLflow Child Run ID (XGBoost): 1202029889e145bca90d843f4d1de6da
2025-04-22 17:25:31,813 - INFO -   Подбор гиперпараметров (2 итерации)...
[I 2025-04-22 17:25:31,815] A new study created in memory with name: XGBoost_Optuna_Parent_c6ff2af256044637b89b265b6baf79de
[I 2025-04-22 17:27:34,007] Trial 0 finished with value: 0.780425218982026 and parameters: {'learning_rate': 0.017020900533811865, 'max_depth': 7, 'subsample': 0.5432266413652306, 'colsample_bytree': 0.8643353126600157, 'gamma': 0.003687146328749473, 'reg_alpha': 0.011648538250251923, 'reg_lambda': 0.003811845662825376, 'min_child_weight': 8}. Best is trial 0 with value: 0.780425218982026.
[I 2025-04-22 17:30:45,872] Trial 1 finished with value: 0.7824548849909099 and parameters: {'learning_rate': 0.020345177107915238, 'max_depth': 5, 'subsample': 0.5675731

🏃 View run XGBoost HPO+CV at: http://82.202.137.136:8000/#/experiments/819865738036629451/runs/1202029889e145bca90d843f4d1de6da
🧪 View experiment at: http://82.202.137.136:8000/#/experiments/819865738036629451


2025-04-22 17:34:29,753 - INFO -   Время обработки XGBoost: 538.62 сек.
2025-04-22 17:34:30,080 - INFO - --- Обработка модели: CatBoost ---
2025-04-22 17:34:30,438 - INFO -   MLflow Child Run ID (CatBoost): 4e056b388a644c90b8195cb7fc5be308
2025-04-22 17:34:30,761 - INFO -   Подбор гиперпараметров (2 итерации)...
[I 2025-04-22 17:34:30,762] A new study created in memory with name: CatBoost_Optuna_Parent_c6ff2af256044637b89b265b6baf79de
[I 2025-04-22 17:38:09,212] Trial 0 finished with value: 0.7792984150512583 and parameters: {'learning_rate': 0.020019661919740264, 'depth': 7, 'l2_leaf_reg': 0.16085746139644125, 'border_count': 194, 'subsample': 0.6593538014944802}. Best is trial 0 with value: 0.7792984150512583.
[I 2025-04-22 17:39:27,827] Trial 1 finished with value: 0.7788009346792029 and parameters: {'learning_rate': 0.07807556558359093, 'depth': 6, 'l2_leaf_reg': 1.1305340475107994, 'border_count': 188, 'subsample': 0.7859644511737276}. Best is trial 0 with value: 0.779298415051258

🏃 View run CatBoost HPO+CV at: http://82.202.137.136:8000/#/experiments/819865738036629451/runs/4e056b388a644c90b8195cb7fc5be308
🧪 View experiment at: http://82.202.137.136:8000/#/experiments/819865738036629451


2025-04-22 17:43:29,500 - INFO -   Время обработки CatBoost: 539.42 сек.


🏃 View run Pipeline Run 2 folds 2 trials at: http://82.202.137.136:8000/#/experiments/819865738036629451/runs/c6ff2af256044637b89b265b6baf79de
🧪 View experiment at: http://82.202.137.136:8000/#/experiments/819865738036629451


2025-04-22 17:43:30,476 - INFO - 5. Обучение финальных моделей на всех данных X, y...
2025-04-22 17:43:30,477 - INFO -   Обучение финальной модели: LightGBM
2025-04-22 17:44:26,389 - INFO -   Модель LightGBM обучена за 55.91 сек.
2025-04-22 17:44:26,392 - INFO -   Обучение финальной модели: XGBoost
2025-04-22 17:45:39,381 - INFO -   Модель XGBoost обучена за 72.99 сек.
2025-04-22 17:45:39,385 - INFO -   Обучение финальной модели: CatBoost
2025-04-22 17:47:10,753 - INFO -   Модель CatBoost обучена за 91.37 сек.
2025-04-22 17:47:10,756 - INFO - --- Пайплайн завершен ---


In [31]:
# Теперь у вас есть обученные модели:
final_model_lgbm = final_models['LightGBM']

print("Финальные модели получены!")

Финальные модели получены!


In [33]:
test_processed.head()

,sk_id_curr,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,amt_goods_price,region_population_relative,days_birth,days_employed,days_registration,days_id_publish,own_car_age,flag_mobil,flag_emp_phone,flag_work_phone,flag_cont_mobile,flag_phone,flag_email,cnt_fam_members,region_rating_client,region_rating_client_w_city,hour_appr_process_start,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,reg_city_not_live_city,reg_city_not_work_city,live_city_not_work_city,ext_source_1,ext_source_2,ext_source_3,apartments_avg,basementarea_avg,years_beginexpluatation_avg,years_build_avg,commonarea_avg,elevators_avg,entrances_avg,floorsmax_avg,floorsmin_avg,landarea_avg,livingapartments_avg,livingarea_avg,nonlivingapartments_avg,nonlivingarea_avg,apartments_mode,basementarea_mode,years_beginexpluatation_mode,years_build_mode,commonarea_mode,elevators_mode,entrances_mode,floorsmax_mode,floorsmin_mode,landarea_mode,livingapartments_mode,livingarea_mode,nonlivingapartments_mode,nonlivingarea_mode,apartments_medi,basementarea_medi,years_beginexpluatation_medi,years_build_medi,commonarea_medi,elevators_medi,entrances_medi,floorsmax_medi,floorsmin_medi,landarea_medi,livingapartments_medi,livingarea_medi,nonlivingapartments_medi,nonlivingarea_medi,totalarea_mode,obs_30_cnt_social_circle,def_30_cnt_social_circle,obs_60_cnt_social_circle,def_60_cnt_social_circle,days_last_phone_change,flag_document_2,flag_document_3,flag_document_4,flag_document_5,flag_document_6,flag_document_7,flag_document_8,flag_document_9,flag_document_10,flag_document_11,flag_document_12,flag_document_13,flag_document_14,flag_document_15,flag_document_16,flag_document_17,flag_document_18,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year,credit_income_ratio,annuity_income_ratio,credit_term,age,years_employed,employed_birth_ratio,age_sq,years_employed_sq,age_emp_interaction,owns_car_and_realty,bureau_loan_count,bureau_loan_types_count,bureau_avg_prolong_count,bureau_total_debt_sum,bureau_max_days_overdue,bureau_overdue_loan_count,prev_app_count,prev_app_approved_count,prev_app_refused_count,prev_app_approved_sum_credit,prev_app_approved_sum_annuity,prev_app_approved_rate,prev_app_approved_avg_credit,prev_app_approved_avg_annuity,inst_payment_diff_mean,inst_payment_diff_max,inst_payment_diff_sum,inst_days_late_mean,inst_days_late_max,inst_days_late_sum,inst_late_payment_count,inst_underpaid_count,inst_paid_on_time_count,inst_total_payment_count,inst_paid_on_time_rate,pos_cash_paid_late_count_recent,pos_cash_paid_late_def_count_recent,pos_cash_avg_instalment_future_recent,pos_cash_completed_count_recent,pos_cash_recent_months_count,cc_balance_avg,cc_balance_max,cc_limit_avg,cc_limit_max,cc_utilization_avg,cc_utilization_max,cc_drawings_atm_avg,cc_drawings_atm_sum,cc_drawings_total_avg,cc_drawings_total_sum,cc_dpd_max,cc_dpd_sum,cc_dpd_def_max,cc_dpd_def_sum,cc_months_count,cc_card_count,bureau_active_debt_sum,curr_credit_over_active_bureau_debt,curr_credit_over_prev_app_credit,income_mean_by_org,income_mean_by_occ,occ_,occ_Accountants,occ_Cleaning staff,occ_Cooking staff,occ_Core staff,occ_Drivers,occ_HR staff,occ_High skill tech staff,occ_IT staff,occ_Laborers,occ_Low-skill Laborers,occ_Managers,occ_Medicine staff,occ_Private service staff,occ_Realty agents,occ_Sales staff,occ_Secretaries,occ_Security staff,occ_Waiters/barmen staff,org_type_freq_encoded,target,name_contract_type_Cash_loans,name_contract_type_Revolving_loans,code_gender_F,code_gender_M,code_gender_XNA,name_type_suite,name_type_suite_Children,name_type_suite_Family,name_type_suite_Group_of_people,name_type_suite_Other_A,name_type_suite_Other_B,name_type_suite_Spouse_partner,name_type_suite_Unaccompanied,name_income_type_Businessman,name_income_type_Commercial_associate,name_income_type_Maternity

In [39]:
# --- Шаг получения предсказаний ---
print("Подготовка данных для предсказания...")
try:
    # Убедись, что переменные test_processed и final_model_lgbm доступны в этой ячейке
    # Если они создаются раньше, то все ок.
    id_col_name = 'sk_id_curr'
    if id_col_name in test_processed.columns:
        # Готовим данные для модели (удаляем ID)
        test_for_prediction = test_processed.drop(columns=[id_col_name])
        print(f"Данные для модели подготовлены. Колонок: {test_for_prediction.shape[1]}") # Ожидаем 310

        # !!! Получаем предсказания и СОЗДАЕМ переменную predictions_lgbm !!!
        print("Получение предсказаний от модели...")
        # Убедись, что final_model_lgbm - это твоя обученная модель LightGBM
        predictions_lgbm = final_model_lgbm.predict_proba(test_for_prediction)[:, 1]
        print("Предсказания успешно получены!")

    else:
        print(f"Ошибка: Колонка '{id_col_name}' не найдена в 'test_processed'.")
        predictions_lgbm = None # Важно для обработки ниже

except NameError as e:
    print(f"ОШИБКА: Переменная не найдена. Убедись, что 'test_processed' и 'final_model_lgbm' определены ранее. {e}")
    predictions_lgbm = None
except Exception as e:
    print(f"\nОШИБКА во время предсказания: {e}")
    predictions_lgbm = None # Важно для обработки ниже
# --- Конец шага получения предсказаний ---

# Проверка, что предсказания получены (опционально, но полезно)
if predictions_lgbm is not None:
    print(f"Создана переменная 'predictions_lgbm', содержащая {len(predictions_lgbm)} предсказаний.")
else:
    print("Переменная 'predictions_lgbm' НЕ была создана из-за ошибки.")

In [41]:

# --- Предсказание и создание сабмишна ---
# ... (код для подготовки test_for_prediction и получения predictions_lgbm) ...

print("Создание файла сабмишна...")
id_col_name = 'sk_id_curr' # Имя колонки ID в твоем test_processed
kaggle_id_col_name = 'SK_ID_CURR' # Имя колонки ID, которое ожидает Kaggle

if id_col_name in test_processed.columns:
    # Берем ID из оригинального test_processed
    submission_df = test_processed[[id_col_name]].copy()

    # --- ИСПРАВЛЕНИЕ ТИПА ДАННЫХ ---
    try:
        # Преобразуем колонку ID в стандартный целый тип Python int
        # Это должно отбросить '.0', если они есть, и обработать как целые числа
        submission_df[id_col_name] = submission_df[id_col_name].astype(int)
        print(f"Тип данных колонки '{id_col_name}' успешно изменен на integer.")

        # Дополнительно можно использовать np.int32, если нужно точно соответствие:
        # submission_df[id_col_name] = submission_df[id_col_name].astype(np.int32)
        # print(f"Тип данных колонки '{id_col_name}' успешно изменен на np.int32.")

    except ValueError as e:
        print(f"ОШИБКА: Не удалось конвертировать колонку '{id_col_name}' в integer: {e}")
        print("Проверьте наличие пропущенных (NaN) или нечисловых значений в этой колонке в test_processed.")
        # Если есть NaN, их нужно обработать перед конвертацией, например, удалить строки или заполнить
        submission_df = None # Прерываем создание сабмишна
    except Exception as e:
        print(f"ОШИБКА при конвертации типа данных: {e}")
        submission_df = None # Прерываем создание сабмишна
    # --- КОНЕЦ ИСПРАВЛЕНИЯ ---

    if submission_df is not None:
        # Переименовываем колонку в 'SK_ID_CURR', если она называется иначе
        if id_col_name != kaggle_id_col_name:
            submission_df = submission_df.rename(columns={id_col_name: kaggle_id_col_name})
            print(f"Колонка ID переименована в '{kaggle_id_col_name}'.")

        # Добавляем предсказания
        prediction_series = pd.Series(predictions_lgbm, index=submission_df.index)

        # Вызываем твою функцию
        final_submission = create_submission(submission_df, prediction_series)

        # Показываем результат и проверяем типы
        print("\nПервые 5 строк файла сабмишна:")
        print(final_submission.head())
        print("\nТипы данных в итоговом файле:")
        print(final_submission.dtypes) # Убедись, что SK_ID_CURR теперь int
        print(f"\nРазмер сабмишна: {final_submission.shape}")

else:
    print(f"Ошибка: Колонка '{id_col_name}' не найдена в 'test_processed'.")

# ... (остальной код, обработка ошибок) ...

Создание файла сабмишна...
Тип данных колонки 'sk_id_curr' успешно изменен на integer.
Колонка ID переименована в 'SK_ID_CURR'.
Файл сабмишна создан: submissions/sub_lgbm_2025-04-22 23-32-21.827110.csv

Первые 5 строк файла сабмишна:
   SK_ID_CURR    target
0      100001  0.028922
1      100005  0.123779
2      100013  0.028123
3      100028  0.036178
4      100038  0.128890

Типы данных в итоговом файле:
SK_ID_CURR      int64
target        float64
dtype: object

Размер сабмишна: (48744, 2)


![kaggle](after.jpg)